<a href="https://www.kaggle.com/code/cartelsmith/iterative-imputer-kmeans-clustering-sauce-profile?scriptVersionId=332996207" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "bbq-sauce-database.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "tobiasvondewitz/bbq-sauce-database-nutrition-flavor-and-ratings",
  file_path,
)

import pandas as pd
import seaborn as sns
sns.set_theme(style='darkgrid')
import warnings
warnings.filterwarnings("ignore", category=Warning)


df = df.drop(columns='source_url')

# General Dataset Info Review

In [ ]:
display(df.head())
print(df.info())

In [ ]:
# Duplicate Rows
print("----"*10)
print("Duplicate Rows")
print("----"*10)
display(df[df.duplicated()])

df = df.drop(df.index[55]) # Dropping duplicate row.

# Missing Values
missing = df.isna().sum()/len(df) * 100

print("----"*10)
print("Missing Values %")
print("----"*10)
print(f'\n{missing.round(2)}')


In [ ]:
drop_cols = ['rating_sentiment', 'opinion_count', 'serving_size',
             'diet_tags', 'pairings','gtin','name','brand']

df = df.drop(drop_cols, axis=1)

In [ ]:
df.info()
print('-+-'*50)
print('---'*50)
print("\nThis dataset has too many missing values. \nAny conclusions drawn via modeling can't be trusted but, for giggles, I'll try to fill them in and build a model.")
print('---'*50)
print('-+-'*50)

# Preprocessing

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing  import MinMaxScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# Imputing
imputer = IterativeImputer(random_state=12)

num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
 #--------------------------------
num_transformer = make_pipeline(IterativeImputer(), MinMaxScaler())
cat_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(sparse_output=False, handle_unknown='ignore'))


# Transforming Columns
col_transformer = make_column_transformer((num_transformer, num_cols),
                                          (cat_transformer, cat_cols))

# Modeling

In [ ]:
# Building Model
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt


model = KMeans(n_clusters=3)


pipe = Pipeline(steps=[('preprocess',col_transformer)])

# Fit the model
df_transformed = pipe.fit_transform(df)

# Finding correct number of clusters
inertia = []

for n in range(1,10):
    kmeans = KMeans(n_clusters=n)
    kmeans.fit(df_transformed)
    inertia.append(kmeans.inertia_)

# Plotting Elbow Chart
sns.pointplot(y=inertia, x=range(1,10), color='Purple')

plt.title("Elbow Plot")
plt.xlabel("Clusters")
plt.ylabel('Inertia')
plt.show();

In [ ]:
print(f"\n\nBased on the Elbow Plot above a reasonable number of clusters is 5-7. I'll use 6.")

# Interpreting the Clusters

In [ ]:
import math
import numpy as np

# 1. Fit the final model with k=6
kmeans_final = KMeans(n_clusters=6, random_state=12)
labels = kmeans_final.fit_predict(df_transformed)

# 2. Attach the new cluster labels to the original dataframe
df['Cluster'] = labels

# 3. Group by cluster and get the mean of the numeric columns
# (Pandas handles the remaining NaNs here automatically by ignoring them in the mean)
cluster_means = df.groupby('Cluster')[num_cols].mean()

# 4. Scale the cluster means so they fit beautifully on the same radar axes (0 to 1 scale)
scaler = MinMaxScaler()
cluster_means_scaled = pd.DataFrame(
    scaler.fit_transform(cluster_means),
    columns=cluster_means.columns,
    index=cluster_means.index
)

# 5. Plotting the Radar Charts
categories = cluster_means_scaled.columns.tolist()
N = len(categories)

# Calculate the angle for each category on the circle
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1] # Close the loop so the chart connects back to the start

# Create a 2x3 grid of polar subplots for our 6 clusters
fig, axes = plt.subplots(2, 3, figsize=(16, 10), subplot_kw=dict(polar=True))
fig.suptitle('BBQ Sauce Cluster Profiles', size=20, y=1.05)
axes = axes.flatten()

# Loop through each cluster and plot its profile
for i, (cluster_id, row) in enumerate(cluster_means_scaled.iterrows()):
    ax = axes[i]
    values = row.tolist()
    values += values[:1] # Close the loop for the values
    
    # Draw the outline and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', color='purple')
    ax.fill(angles, values, 'purple', alpha=0.25)
    
    # Add category labels around the perimeter
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=9)
    
    # Clean up the aesthetics
    ax.set_title(f'Cluster {cluster_id}', size=14, y=1.1, weight='bold')
    ax.set_yticks([]) # Hide the concentric circular grid numbers for a cleaner look
    ax.spines['polar'].set_visible(False) # Remove the outer border ring

plt.tight_layout()
plt.show()